<a href="https://colab.research.google.com/github/UmymaM/ml-dl-cv-fundamentals/blob/main/face-verification/face_verification_w_Siamese_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

In [52]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


In [53]:
!kaggle datasets download -d atulanandjha/lfwpeople

Dataset URL: https://www.kaggle.com/datasets/atulanandjha/lfwpeople
License(s): GNU Lesser General Public License 3.0
lfwpeople.zip: Skipping, found more recently modified local copy (use --force to force download)


In [54]:
import zipfile
zipfile=zipfile.ZipFile('lfwpeople.zip') #unzipping the lfwpeople file
zipfile.extractall()
zipfile.close()

In [55]:
import tarfile

# Path to the tgz file
tgz_file_path = '/content/lfw-funneled.tgz'

# Directory to extract to
extract_dir = '/content/'

# Open the tgz file in read mode ('r:gz')
with tarfile.open(tgz_file_path, 'r:gz') as tar:
    # Extract all contents to the specified directory
    tar.extractall(path=extract_dir)

/tmp/ipykernel_3288/4013497105.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


In [56]:
train_transform=transforms.Compose([
    transforms.Resize((112,112)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])
val_transform=transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor(), #no augmentation for test ds
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [57]:
import os

In [58]:
labeled_faces_dict={}
# filtering files
for file in os.listdir('/content/lfw_funneled'):
    file_path = os.path.join('/content/lfw_funneled', file)
    images=[]
    if os.path.isdir(file_path):
        # print(f"Length of dir {file}: {len(os.listdir(file_path))}")
        # continue
        for image in os.listdir(file_path):
          image=os.path.join(file_path,image)
          images.append(image)
        labeled_faces_dict[file]=images
    else:
        print(f"Skipping file: {file}")

Skipping file: pairs_07.txt
Skipping file: pairs_02.txt
Skipping file: pairs.txt
Skipping file: pairs_08.txt
Skipping file: pairs_04.txt
Skipping file: pairs_03.txt
Skipping file: pairs_10.txt
Skipping file: pairs_01.txt
Skipping file: pairs_06.txt
Skipping file: pairs_05.txt
Skipping file: pairs_09.txt


In [59]:
len(labeled_faces_dict.keys())

5749

In [60]:
labeled_faces_dict.get("Charles_Schumer")

['/content/lfw_funneled/Charles_Schumer/Charles_Schumer_0002.jpg',
 '/content/lfw_funneled/Charles_Schumer/Charles_Schumer_0001.jpg']

In [61]:
for idx,file in enumerate(labeled_faces_dict.keys()):
  print(f"{idx}: {file} {len(labeled_faces_dict[file])}")
  if idx==20:
    break

0: Jimmy_Kimmel 2
1: Alma_Powell 1
2: Jacques_Chirac 52
3: Brittany_Snow 1
4: Terry_Gilliam 1
5: Cathryn_Crawford 1
6: Ibrahim_Hilal 1
7: Mario_Vasquez_Rana 1
8: Yannos_Papantoniou 1
9: Tom_Kelly 1
10: Linda_Mason 1
11: Horacio_Julio_Pina 1
12: Perry_Compton 1
13: Tommy_Shane_Steiner 1
14: Wally_Szczerbiak 1
15: Jonathan_Edwards 8
16: Debbie_Allen 1
17: Benjamin_Bratt 1
18: Daniel_Pearl 1
19: Jerome_Golmard 1
20: David_Alpay 1


In [62]:
from itertools import combinations

In [63]:
positive_pairs=[]
negative_pairs=[]
# making +ve pairs
for person,images in labeled_faces_dict.items():
  if len(images)>1:
      for image1,image2 in combinations(images,2):
        pair=(image1,image2,1)
        positive_pairs.append(pair)

In [64]:
for i in range(10):
  print(f"Positive Pairs: {positive_pairs[i]}")

Positive Pairs: ('/content/lfw_funneled/Jimmy_Kimmel/Jimmy_Kimmel_0002.jpg', '/content/lfw_funneled/Jimmy_Kimmel/Jimmy_Kimmel_0001.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0012.jpg', '/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0010.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0012.jpg', '/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0008.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0012.jpg', '/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0044.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0012.jpg', '/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0042.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0012.jpg', '/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0015.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Jacques_Chirac/Jacques_Chirac_0012.jpg', '/content/lfw_funneled/Jacques_Chirac/J

In [65]:
import random

In [66]:
type(labeled_faces_dict.keys())

dict_keys

In [67]:
# making negative pairs
# for a balanced ds, we'll iterate the loop the same number of times as there are +ve pairs
for i in range(len(positive_pairs)):
  # generating 2 random names from the dict
  n1,n2=random.sample(list(labeled_faces_dict.keys()),2)
  image1=random.choice(labeled_faces_dict[n1])
  image2=random.choice(labeled_faces_dict[n2])
  pair=(image1,image2,0)
  negative_pairs.append(pair)

In [68]:
for i in range(10):
  print(f"Negative Pairs: {negative_pairs[i]}")

Negative Pairs: ('/content/lfw_funneled/Sven_Goran_Eriksson/Sven_Goran_Eriksson_0001.jpg', '/content/lfw_funneled/Darrell_Issa/Darrell_Issa_0002.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Edward_Said/Edward_Said_0002.jpg', '/content/lfw_funneled/Thomas_Ferguson/Thomas_Ferguson_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Din_Samsudin/Din_Samsudin_0001.jpg', '/content/lfw_funneled/Gerald_Fitch/Gerald_Fitch_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Rainer_Geulen/Rainer_Geulen_0001.jpg', '/content/lfw_funneled/Kate_Moss/Kate_Moss_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Robert_McKee/Robert_McKee_0001.jpg', '/content/lfw_funneled/Saied_Hadi_al_Mudarissi/Saied_Hadi_al_Mudarissi_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Sterling_Hitchcock/Sterling_Hitchcock_0001.jpg', '/content/lfw_funneled/Natalie_Coughlin/Natalie_Coughlin_0003.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Norodom_Sihanouk/Norodom_Sihanouk_0001.jpg', '/content/lfw_funne

In [69]:
print(len(positive_pairs))
print(len(negative_pairs))

242257
242257


In [70]:
combined_pairs=positive_pairs+negative_pairs
for i in range(20):
  print(f"Combined Pairs: {combined_pairs[-i]}")

Combined Pairs: ('/content/lfw_funneled/Jimmy_Kimmel/Jimmy_Kimmel_0002.jpg', '/content/lfw_funneled/Jimmy_Kimmel/Jimmy_Kimmel_0001.jpg', 1)
Combined Pairs: ('/content/lfw_funneled/Anthony_Principi/Anthony_Principi_0001.jpg', '/content/lfw_funneled/Ray_Bradbury/Ray_Bradbury_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Chris_Rock/Chris_Rock_0001.jpg', '/content/lfw_funneled/John_Fox/John_Fox_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Donna_Shalala/Donna_Shalala_0001.jpg', '/content/lfw_funneled/Mike_Scioscia/Mike_Scioscia_0002.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Bill_OReilly/Bill_OReilly_0001.jpg', '/content/lfw_funneled/Juan_Pablo_Montoya/Juan_Pablo_Montoya_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Luiz_Inacio_Lula_da_Silva/Luiz_Inacio_Lula_da_Silva_0046.jpg', '/content/lfw_funneled/Leslie_Wiser_Jr/Leslie_Wiser_Jr_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Mike_Farrar/Mike_Farrar_0001.jpg', '/content/lfw_funneled/Roger_Penske/Roger

In [71]:
random.shuffle(combined_pairs)

In [72]:
from PIL import Image

In [73]:
# for index,pair in enumerate(combined_pairs):
#     img1=train_transform(Image.open(pair[0]))
#     img2=train_transform(Image.open(pair[1]))
#     label=pair[2]
#     combined_pairs[index]=(img1, img2, label)

In [74]:
from torch.utils.data import Dataset
from PIL import Image

In [75]:
class SiameseDataset(Dataset):
  def __init__(self,pairs,transform=None) -> None:
     self.pairs=pairs
     self.transform=transform
  def __len__(self):
    return len(self.pairs)
  def __getitem__(self,index):
    img1,img2,label=self.pairs[index]
    img1=Image.open(img1).convert("RGB")
    img2=Image.open(img2).convert("RGB")
    if self.transform:
      img1=self.transform(img1)
      img2=self.transform(img2)
      return img1,img2,torch.tensor(label,dtype=torch.float32)


In [76]:
total=len(combined_pairs)
train_size=int(0.8 * total)
val_size=total-train_size

In [77]:
train_pairs=combined_pairs[:train_size]
val_pairs=combined_pairs[train_size:]

In [78]:
train_dataset=SiameseDataset(train_pairs,transform=train_transform)
val_dataset=SiameseDataset(val_pairs,transform=val_transform)

In [79]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=4,pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=4,pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [80]:
print(type(train_loader.dataset[1][0]))

<class 'torch.Tensor'>


In [81]:
img1,img2,label=train_dataset[1]
print(f"Image 1: {img1.shape}")
print(f"Image 2: {img2.shape}")
print(f"Label: {label}")

Image 1: torch.Size([3, 112, 112])
Image 2: torch.Size([3, 112, 112])
Label: 1.0


In [82]:
for img1,img2,label in train_loader:
  print(img1.shape)
  print(img2.shape)
  print(label.shape)
  break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


torch.Size([32, 3, 112, 112])
torch.Size([32, 3, 112, 112])
torch.Size([32])


In [83]:
class SiameseNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.backbone=models.resnet18(pretrained=True)
    self.backbone.fc=nn.Linear(self.backbone.fc.in_features,128)
  def forward(self,pair1,pair2):
    emb1=self.backbone(pair1)
    emb2=self.backbone(pair2)
    return emb1,emb2

In [84]:
class ContrastiveLoss(nn.Module):
  def __init__(self,margin=1.0):
    super().__init__()
    self.margin=margin
  def forward(self,emb1,emb2,label):
    # calculating the euclidean distance bw the embeddings
    distance=torch.pairwise_distance(emb1,emb2)
    # formula for contrastive loss: L=Y⋅D2+(1−Y)⋅max(0,m−D)2
    loss = torch.mean(label*torch.pow(distance,2)+(1-label)*
                      torch.pow(torch.clamp(self.margin-distance,min=0.0),2))
    return loss

In [85]:
model=SiameseNetwork()
model.to(device=device)
criterion=ContrastiveLoss()
optimizer=optim.Adam(model.parameters(),lr=0.0001)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


In [ ]:
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()
for epoch in range(5):
    model.train()
    total_loss = 0
    for img1,img2,label in train_loader:
        img1=img1.to(device)
        img2=img2.to(device)
        label=label.float().to(device)
        optimizer.zero_grad()
        with autocast():
            emb1, emb2 = model(img1, img2)
            loss = criterion(emb1, emb2, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss+=loss.item()

    avg_loss=total_loss/len(train_loader)
    print(f"Epoch {epoch+1}/5  Loss: {avg_loss:.2f}")


/tmp/ipykernel_3288/814346814.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_3288/814346814.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/5  Loss: 0.02
Epoch 2/5  Loss: 0.01
